In [ ]:
import asyncio
import os
from google import genai
from google.genai import types
# 1. Define a tool the model can call
def get_current_time():
    """Returns the current system time."""
    from datetime import datetime
    return {"time": datetime.now().strftime("%H:%M:%S")}

# 2. Setup Client
# Ensure you use 'v1alpha' for preview features
client = genai.Client(
    api_key="YOUR_API_KEY", 
    http_options={'api_version': 'v1alpha'}
)

MODEL_ID = "gemini-2.5-flash-native-audio-preview-12-2025"

async def main():
    # 3. Configure the session for audio + text + tools
    config = types.LiveConnectConfig(
        tools=[get_current_time],
        response_modalities=["AUDIO"], # Returns native audio
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        # This returns the text transcript of the model's audio response
        output_audio_transcription=types.AudioTranscriptionConfig()
    )

    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
        # await session.send_tool_response()
        # Send text input or audio bytes (e.g., from a mic stream)
        # To send audio, replace the string with audio bytes:
        # await session.send(audio_bytes, end_of_turn=True)
        await session.send_client_content(turns= types.Content(role="user", parts=[types.Part(text="What time is it?")]), turn_complete=True)

        async for message in session.receive():
            # A. HANDLE FUNCTION CALLS
            # message.tool_call.function_calls
            # message.server_content.model_turn.parts
            message.server_content.output_transcription.


            if message.tool_call:
                if message.tool_call.function_calls:
                    for call in message.tool_call.function_calls:
                        print(f"Executing: {call.name}")
                        await session.send(
                            types.LiveClientToolResponse(
                                function_responses=[
                                    types.FunctionResponse(name=call.name, id=call.id, response=result)
                                ]
                            )
                        )

            # B. HANDLE TEXT & AUDIO OUTPUT
            if message.server_content and message.server_content.model_turn:
                for part in message.server_content.model_turn.parts:
                    # Capture Text Transcript
                    if part.text:
                        print(f"Model Transcript: {part.text}")
                    
                    # Capture Audio Data
                    if part.inline_data:
                        # part.inline_data.data contains the base64 audio bytes
                        print("Received audio chunk...")

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import time
print(time.time())

In [3]:
l = [1,2,3,4,5]
print(l[:2])

[1, 2]
